# Demo implementation of all algorithms

In [1]:
# import necessary libraries
import numpy as np
from simulate_data import simulate_views

## Simulate data

In [2]:
matrices = simulate_views(n=500, num_genes=20, M_C=20, M_Z=3, rank=3, seed=0)
print(matrices.keys())


dict_keys(['G', 'C', 'Z', 'W_C', 'W_G', 'H_G', 'H_C', 'U_G', 'U_C'])


## Run SCoNE

In [ ]:
from algorithms.SCoNE import SCoNE_parallel

factor_matrices, loss_function = SCoNE_parallel(
    matrices['G'].astype(float),matrices['C'].astype(float),Z, rank=3,
    alpha=max(matrices['G'].max(),matrices['C'].max())**2,lambda_H_G=1e-4, lambda_H_C=1e-4, lambda_Gloss=1,   # regularization parameters
    num_init=50,init='random',G_loss_type='kl_div', C_loss_type='kl_div', # in 'kl_div',  'fro' or None
    n_jobs=20
)

In [11]:
from evaluation.reconstruction_evaluation import best_permutation_similarity

best_permutation_similarity(matrices['W_C'],factor_matrices['W'])

0.7294098201159154

In [7]:
best_permutation_similarity(matrices['U_G'],factor_matrices['U_G'])

0.7102623502990604

## Run SCoNE (Fro)

In [72]:
factor_matrices, loss_function = SCoNE_parallel(
    matrices['G'],matrices['C'],matrices['Z'], rank=3,
    alpha=max(matrices['G'].max(),matrices['C'].max())**2,lambda_H_G=1e-4, lambda_H_C=1e-4, lambda_Gloss=1,         # regularization parameters
    num_init=10,init='nndsvda',G_loss_type='fro', C_loss_type='fro', # in 'kl_div',  'fro' or None
)
best_permutation_similarity(matrices['W_C'],factor_matrices['W'])

0.7127066270253228

## Run CoNE

In [73]:
factor_matrices, loss_function = SCoNE_parallel(
    matrices['G'],matrices['C'],matrices['Z'], rank=3,
    alpha=0,lambda_H_G=0, lambda_H_C=0, lambda_Gloss=1,         # regularization parameters
    num_init=10,init='nndsvda',G_loss_type='kl_div', C_loss_type='kl_div', # in 'kl_div',  'fro' or None
)
best_permutation_similarity(matrices['W_C'],factor_matrices['W'])

0.7994121517008319

## Run HNMF

In [74]:
factor_matrices, loss_function = SCoNE_parallel(
    matrices['G'],matrices['C'],None, rank=3,
    alpha=0,lambda_H_G=0, lambda_H_C=0, lambda_Gloss=1,         # regularization parameters
    num_init=10,init='nndsvda',G_loss_type='kl_div', C_loss_type='kl_div', # in 'kl_div',  'fro' or None
)
best_permutation_similarity(matrices['W_C'],factor_matrices['W'])

0.7418505950605683

## Run HNMF (res)

In [75]:
from algorithms.SCoNE import proj_nonneg

C_resid = proj_nonneg(matrices['C'] - matrices['Z'] @ np.linalg.lstsq(matrices['Z'], matrices['C'],rcond=None)[0])
G_resid = proj_nonneg(matrices['G'] - matrices['Z'] @ np.linalg.lstsq(matrices['Z'], matrices['G'],rcond=None)[0])

factor_matrices, loss_function = SCoNE_parallel(
    G_resid,C_resid,None, rank=3,
    alpha=0,lambda_H_G=0, lambda_H_C=0, lambda_Gloss=1,         # regularization parameters
    num_init=10,init='nndsvda',G_loss_type='fro', C_loss_type='fro', # in 'kl_div',  'fro' or None
)
best_permutation_similarity(matrices['W_C'],factor_matrices['W'])

0.7007339385522637

## Run C-CoNE

In [76]:
factor_matrices, loss_function = SCoNE_parallel(
    None,matrices['C'],matrices['Z'], rank=3,
    num_init=10,init='nndsvda',G_loss_type=None, C_loss_type='kl_div', # in 'kl_div',  'fro' or None
)
best_permutation_similarity(matrices['W_C'],factor_matrices['W'])

0.6959518784798652

## Run G-CoNE

In [77]:
factor_matrices, loss_function = SCoNE_parallel(
    matrices['G'],None,matrices['Z'], rank=3,
    num_init=10,init='nndsvda',G_loss_type='kl_div', C_loss_type=None, # in 'kl_div',  'fro' or None
)
best_permutation_similarity(matrices['W_C'],factor_matrices['W'])

0.6945459226410914

## Run C-NMF

In [78]:
factor_matrices, loss_function = SCoNE_parallel(
    None,matrices['C'],None, rank=3,
    num_init=10,init='nndsvda',G_loss_type=None, C_loss_type='kl_div', # in 'kl_div',  'fro' or None
)
best_permutation_similarity(matrices['W_C'],factor_matrices['W'])

0.6666500625689084

## Run G-NMF

In [79]:
factor_matrices, loss_function = SCoNE_parallel(
    matrices['G'],None,None, rank=3,
    num_init=10,init='nndsvda',G_loss_type='kl_div', C_loss_type=None, # in 'kl_div',  'fro' or None
)
best_permutation_similarity(matrices['W_C'],factor_matrices['W'])

0.6487308422655256

## Run RGWAS

In [6]:
matrices["C"]

array([[2., 0., 0., ..., 0., 1., 0.],
       [0., 0., 4., ..., 0., 0., 0.],
       [0., 0., 1., ..., 0., 0., 0.],
       ...,
       [1., 0., 0., ..., 0., 1., 0.],
       [1., 0., 0., ..., 1., 1., 0.],
       [2., 0., 2., ..., 3., 2., 1.]])

In [3]:
import os
import pandas as pd
from algorithms.RGWASWrapper import RGWASWrapper

cwd = os.getcwd()
parent_dir = os.path.dirname(os.getcwd())
# NOTE: important to save as float64 so read correctly in R
pd.DataFrame(matrices["G"]).to_csv(f'{parent_dir}/example_data/G.csv') 
pd.DataFrame(matrices["C"]).to_csv(f'{parent_dir}/example_data/C.csv') 
pd.DataFrame(matrices["Z"]).to_csv(f'{parent_dir}/example_data/Z.csv') 
factor_matrices, loss_function = RGWASWrapper(
    r_path='/gpfs/commons/home/anewbury/miniconda/bin/Rscript', # REPLACE WITH CORRECT Rscript path
    G_path=f'{parent_dir}/example_data/G.csv',
    C_path=f'{parent_dir}/example_data/C.csv',
    Z_path=f'{parent_dir}/example_data/Z.csv',
    rank=5, num_init=10, write_all_init=False)
best_permutation_similarity(matrices['W_C'],factor_matrices['W'])

Error in outs[[which.min(lls)]] : 
  attempt to select less than one element in get1index
Calls: mfmr
Execution halted


JSONDecodeError: Expecting value: line 1 column 1 (char 0)

## Run MVBC

In [82]:
import algorithms.MVBCWrapper as MVBCWrapper
import importlib
importlib.reload(MVBCWrapper)

factor_matrices, loss_function = MVBCWrapper.MVBCWrapper(
    G_path=f'{parent_dir}/example_data/G.npy', C_path=f'{parent_dir}/example_data/C.npy', 
    rank=3, lambda_W=1, lambda_H_G=1, lambda_H_C=1,  
    r_path='/gpfs/commons/home/anewbury/miniconda/bin/Rscript') # REPLACE WITH CORRECT Rscript path
best_permutation_similarity(matrices['W_C'],factor_matrices['W'])

0.42211615474295544